# CIFAR-10 종합 프로젝트

**30일 딥러닝 완전 정복 · Day 21 · 3주차 프로젝트** — 강의 본문: [https://muy-bien-hwa.github.io/For-my-Claude/](https://muy-bien-hwa.github.io/For-my-Claude/)

### 이 노트북에서 할 일
- Day 16 템플릿 + Day 18 ResNet + Day 19 증강을 조합한다
- 목표: 테스트 정확도 93% 이상, 클래스별 정확도·오분류 분석
- 보너스: 사전학습 ResNet 미세조정으로 96%+

> ⚙️ **런타임 → 런타임 유형 변경 → T4 GPU** 를 먼저 선택하세요. 셀은 위에서부터 `Shift+Enter`로 실행합니다.
> 그래프 라벨은 Colab 기본 글꼴 문제를 피하려고 영어로 표기했습니다.

In [ ]:
import random, os, json, time
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from tqdm.auto import tqdm
device = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def train_one_epoch(model, loader, loss_fn, opt, scheduler=None, clip=None, scaler=None):
    model.train(); tot, correct, n = 0.0, 0, 0
    for xb, yb in tqdm(loader, leave=False):
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=scaler is not None):
            logits = model(xb); loss = loss_fn(logits, yb)
        opt.zero_grad(set_to_none=True)
        if scaler is not None:
            scaler.scale(loss).backward()
            if clip: scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(opt); scaler.update()
        else:
            loss.backward()
            if clip: torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
        if scheduler is not None: scheduler.step()
        tot += loss.item() * len(yb); correct += (logits.argmax(1) == yb).sum().item(); n += len(yb)
    return tot / n, correct / n

@torch.no_grad()
def evaluate(model, loader, loss_fn):
    model.eval(); tot, correct, n = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb); tot += loss_fn(logits, yb).item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item(); n += len(yb)
    return tot / n, correct / n

def save_ckpt(path, model, opt, epoch, best):
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "epoch": epoch, "best": best}, path)

def load_ckpt(path, model, opt=None):
    ck = torch.load(path, map_location=device)
    model.load_state_dict(ck["model"])
    if opt is not None: opt.load_state_dict(ck["opt"])
    return ck["epoch"], ck["best"]

def fit(model, train_dl, val_dl, epochs, lr=1e-3, wd=0.05, ckpt="best.pt", patience=None, label_smoothing=0.0, amp=True, sgd=False):
    loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    if sgd: opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True, weight_decay=wd)
    else:   opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(train_dl), pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda") if (amp and device == "cuda") else None
    hist = {k: [] for k in ["tr_loss", "tr_acc", "va_loss", "va_acc"]}; best, bad = 0.0, 0
    for ep in range(epochs):
        t0 = time.time()
        tl, ta = train_one_epoch(model, train_dl, loss_fn, opt, sched, clip=1.0, scaler=scaler)
        vl, va = evaluate(model, val_dl, nn.CrossEntropyLoss())
        for k, v in zip(hist, [tl, ta, vl, va]): hist[k].append(v)
        print(f"[{ep+1:02d}/{epochs}] train {tl:.4f}/{ta:.3f}  val {vl:.4f}/{va:.3f}  lr {opt.param_groups[0]['lr']:.2e}  {time.time()-t0:.0f}s")
        if va > best: best, bad = va, 0; save_ckpt(ckpt, model, opt, ep, best)
        else:
            bad += 1
            if patience and bad >= patience: print("early stopping"); break
    load_ckpt(ckpt, model)
    return hist

def plot_hist(hist, title=""):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
    ax[0].plot(hist["tr_loss"], label="train"); ax[0].plot(hist["va_loss"], label="val"); ax[0].set_title(f"{title} loss"); ax[0].legend()
    ax[1].plot(hist["tr_acc"], label="train"); ax[1].plot(hist["va_acc"], label="val"); ax[1].set_title(f"{title} accuracy"); ax[1].legend()
    plt.show()
print("template ready on", device)

## 1. 데이터 (강한 증강)

In [ ]:
mean, std = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
train_tf = transforms.Compose([transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(),
                               transforms.TrivialAugmentWide(), transforms.ToTensor(), transforms.Normalize(mean, std),
                               transforms.RandomErasing(p=0.25)])
test_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])
tr_full = datasets.CIFAR10("data", train=True, download=True, transform=train_tf)
va_full = datasets.CIFAR10("data", train=True, download=True, transform=test_tf)
test_ds = datasets.CIFAR10("data", train=False, download=True, transform=test_tf)
idx = torch.randperm(50000, generator=torch.Generator().manual_seed(0))
train_dl = DataLoader(torch.utils.data.Subset(tr_full, idx[:45000]), 256, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_dl = DataLoader(torch.utils.data.Subset(va_full, idx[45000:]), 1000, num_workers=2)
test_dl = DataLoader(test_ds, 1000, num_workers=2); classes = tr_full.classes

## 2. 모델 (Day 18의 ResNet)

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, c_in, c_out, stride=1):
        super().__init__()
        self.c1 = nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(c_out)
        self.c2 = nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False); self.b2 = nn.BatchNorm2d(c_out)
        self.sc = nn.Identity() if (stride == 1 and c_in == c_out) else nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm2d(c_out))
    def forward(self, x): return F.relu(self.b2(self.c2(F.relu(self.b1(self.c1(x))))) + self.sc(x))
class ResNet18(nn.Module):
    def __init__(self, n=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, 64, 3, 1, 1, bias=False), nn.BatchNorm2d(64), nn.ReLU())
        L, c = [], 64
        for i, w in enumerate([64, 128, 256, 512]):
            for j in range(2): L.append(BasicBlock(c, w, 2 if (j == 0 and i > 0) else 1)); c = w
        self.layers = nn.Sequential(*L); self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(512, n))
    def forward(self, x): return self.head(self.layers(self.stem(x)))

## 3. 배치 하나 과적합 테스트 (Day 28) → 본 학습

In [ ]:
seed_everything(0); m = ResNet18().to(device); o = torch.optim.AdamW(m.parameters(), 1e-3)
xb, yb = next(iter(train_dl)); xb, yb = xb[:32].to(device), yb[:32].to(device)
for i in range(150):
    l = F.cross_entropy(m(xb), yb); o.zero_grad(); l.backward(); o.step()
print("single-batch loss after 150 steps:", l.item(), "(should be near 0)")

In [ ]:
seed_everything(42)
model = ResNet18().to(device).to(memory_format=torch.channels_last)
hist = fit(model, train_dl, val_dl, epochs=40, lr=0.1, wd=5e-4, sgd=True, label_smoothing=0.1, ckpt="proj_best.pt")
plot_hist(hist, "CIFAR-10 project")

## 4. 최종 평가와 분석

In [ ]:
import matplotlib.pyplot as plt
test_loss, test_acc = evaluate(model, test_dl, nn.CrossEntropyLoss())
print(f"TEST accuracy {test_acc:.4f}")
model.eval(); preds, ys, probs, xs = [], [], [], []
with torch.no_grad():
    for xb, yb in test_dl:
        p = F.softmax(model(xb.to(device)), -1).cpu(); preds.append(p.argmax(1)); probs.append(p.max(1).values); ys.append(yb); xs.append(xb)
preds, ys, probs, xs = map(torch.cat, (preds, ys, probs, xs))
cm = torch.zeros(10, 10, dtype=torch.long)
for t, p in zip(ys, preds): cm[t, p] += 1
for i, c in enumerate(classes): print(f"{c:10s} {cm[i, i].item() / cm[i].sum().item():.3f}")
plt.figure(figsize=(6, 5)); plt.imshow(cm, cmap="Blues"); plt.xticks(range(10), classes, rotation=60); plt.yticks(range(10), classes); plt.colorbar(); plt.show()

wrong = torch.where(preds != ys)[0]; top = wrong[probs[wrong].argsort(descending=True)[:16]]
fig, axs = plt.subplots(2, 8, figsize=(16, 4.5))
for a, i in zip(axs.ravel(), top):
    img = xs[i].permute(1, 2, 0) * torch.tensor(std) + torch.tensor(mean)
    a.imshow(img.clamp(0, 1)); a.set_title(f"{classes[ys[i]]}->{classes[preds[i]]}\n{probs[i]:.2f}", fontsize=8); a.axis("off")
plt.suptitle("most confident mistakes"); plt.show()

## 5. 보너스: 사전학습 모델 미세조정 (96%+)
ImageNet 사전학습 ResNet-50을 128×128로 키운 이미지에 미세조정합니다. T4에서 약 25분.

In [ ]:
from torchvision import models
up_tr = transforms.Compose([transforms.Resize(128), transforms.RandomCrop(128, padding=8), transforms.RandomHorizontalFlip(),
                            transforms.ToTensor(), transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
up_te = transforms.Compose([transforms.Resize(128), transforms.ToTensor(), transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
tr_b = DataLoader(torch.utils.data.Subset(datasets.CIFAR10("data", True, transform=up_tr), idx[:45000]), 128, shuffle=True, num_workers=2, drop_last=True)
va_b = DataLoader(torch.utils.data.Subset(datasets.CIFAR10("data", True, transform=up_te), idx[45000:]), 500, num_workers=2)
pt = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2); pt.fc = nn.Linear(2048, 10); pt = pt.to(device)
hist_pt = fit(pt, tr_b, va_b, epochs=5, lr=3e-4, wd=0.05, ckpt="pt_best.pt")

## 과제
- 결과 표(베이스라인 / ResNet18 / 미세조정)를 학습 노트에 기록하세요.
- 가장 많이 헷갈린 클래스 쌍은 무엇이고, 왜 그럴까요?